In [1]:
import pandas as pd
import numpy as np
import spacy

In [2]:
df = pd.read_csv('movie3.csv')
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])
df

,text,label
0,Eugene O'Neill is acclaimed by some as America...,0
1,"OK, before I get into this, let's go ahead and...",0
2,John Wayne is without a doubt one of the most ...,0
3,Uzumaki has a formidable reputation within Lov...,1
4,"I really liked ZB1. Really, I did. I have no p...",0
...,...,...
9995,I saw this movie on TV and loved it! I am a re...,1
9996,"I used to watch this show as a child, and I lo...",1
9997,The Great Dictator is a beyond-excellent film....,1
9998,This has got to be the worst show I have ever ...,0


In [3]:
nlp = spacy.load("en_core_web_sm")

def preprocess_text(sentence):
    doc = nlp(sentence)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

In [4]:
processed_sentences = [preprocess_text(str(sent)) for sent in df["text"]]

In [5]:
processed_sentences = pd.Series(processed_sentences)
processed_sentences

0       Eugene O'Neill acclaim America lead playwright...
1       ok let ahead warning way movie quintessential ...
2       John Wayne doubt popular loved actor time care...
3       Uzumaki formidable reputation lovecraftian cir...
4       like zb1 problem extremely low budget movie en...
                              ...                        
9995    see movie tv love real disaster film fan great...
9996    watch child love watch come Toonami year ago S...
9997    Great Dictator excellent film Charlie Chaplin ...
9998    get bad see like Chuck Norris Films need show ...
9999    Good Lord think spoiler warning think matter w...
Length: 10000, dtype: object

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

In [7]:
# Розбиття даних на тренувальні та тестові
X_train, X_test, y_train, y_test = train_test_split(processed_sentences, df['label'], test_size=0.2)

# Векторизація речень
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [8]:
# Логістична регресія
lr = LogisticRegression(penalty='l2', max_iter=1000, C=1)
lr.fit(X_train_vec, y_train)

LogisticRegression(C=1, max_iter=1000)

In [9]:
y_pred = lr.predict(X_test_vec)

print("Точність моделі:", accuracy_score(y_test, y_pred))
print("Матриця невідповідностей:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

Точність моделі: 0.8505
Матриця невідповідностей:
[[785 169]
 [130 916]]


In [12]:
from textblob import TextBlob

# Класифікація з використанням TextBlob
predicted_labels = processed_sentences.apply(lambda x: 1 if TextBlob(x).sentiment.polarity > 0 else 0)

print("Точність моделі:", accuracy_score(df['label'], predicted_labels))
print("Матриця невідповідностей:")
cm = confusion_matrix(df['label'], predicted_labels)
print(cm)

Точність моделі: 0.7151
Матриця невідповідностей:
[[2621 2384]
 [ 465 4530]]


In [17]:
ids = [np.random.randint(0, processed_sentences.shape[0]) for _ in range(3)]
random_sentences = [processed_sentences.iloc[idx] for idx in ids]
random_sentences

['fan new movie real pleasure watch adult child watch unusual aunt like laughter tear love adventure special effect good actor talk parrot remind favourite Wizard Oz hero Paulie intelligent parrot separate home family go adventure temptation disappointment keep mind resolution find friend Marie highly recommend',
 'ah man movie funny hell strange like keep shakespearian language movie feel ironic idiotic movie movie get troma good movie highly recommend senseless fun',
 'bad movie absolutely boring wonder shoot dark Vincent Gallo walk away camera look want face actually agree movie like early stuff pretentiousness Brown Bunny cast drivel knock peg book Val Kilmer actually want send sympathy card door actually casting reason watch movie Val movie halfway simply sad movie']

In [18]:
random_sentences_vec = vectorizer.transform(random_sentences)
random_sentences_pred = lr.predict(random_sentences_vec)

print('Передбачені настрої:', random_sentences_pred)
print('Справжні настрої:', [df.loc[idx, 'label'] for idx in ids])

Передбачені настрої: [1 1 0]
Справжні настрої: [1, 1, 0]


In [19]:
random_predicted_labels = [1 if TextBlob(sent).sentiment.polarity > 0 else 0 for sent in random_sentences]

print('Передбачені настрої:', random_predicted_labels)
print('Справжні настрої:', [df.loc[idx, 'label'] for idx in ids])

Передбачені настрої: [1, 1, 0]
Справжні настрої: [1, 1, 0]
